In [15]:

import os

# Create data folder
os.makedirs("data", exist_ok=True)

# Create at least 15 records (including 3 faulty records)
records = [
    "WO01, Machine A, 08:00, Completed",
    "WO02, Machine B, 08:15, Completed",
    "WO03, Machine C, 08:30, Pending",
    "WO04, Machine A, 08:45, Completed",
    "WO05, Machine B, 09:00, Completed",
    "WO06, Machine C, 09:15, Pending",
    "WO07, Machine A, 09:30, Completed",
    "WO08, Machine B, 09:45, Completed",
    "WO09, Machine C, 10:00, Completed",
    "WO10, Machine A, 10:15, Pending",
    "WO11, Machine B, 10:30, Completed",
    "WO12, Machine C, 10:45, Completed",

    # Three deliberately faulty records
    "WO13, Machine A, INVALID_TIME, Completed",
    "WO14, Machine B, 11:15, ",
    "WO15, , 11:30, Completed"
]

# Write records into the file
with open("data/shift_log.txt", "w") as file:
    for record in records:
        file.write(record + "\n")

print("Shift log file created successfully!")

# Read the file and count lines
with open("data/shift_log.txt", "r") as file:
    lines = file.readlines()

print("Total number of records:", len(lines))


Shift log file created successfully!
Total number of records: 15


In [16]:

import os
import csv

# Create data folder if it does not exist
os.makedirs("data", exist_ok=True)

input_file = "data/shift_log.txt"
report_file = "data/report.txt"
reject_file = "data/rejects.csv"

valid_records = []
rejected_records = []

# STEP 1: Read the input file
with open(input_file, "r") as file:
    lines = file.readlines()

# STEP 2: Parse and validate records
for line_number, line in enumerate(lines, start=1):

    line = line.strip()

    if not line:
        rejected_records.append([line_number, line, "Empty line"])
        continue

    parts = line.split(",")

    # Check number of fields
    if len(parts) != 4:
        rejected_records.append([line_number, line, "Incorrect number of fields"])
        continue

    wo, machine, time, status = [p.strip() for p in parts]

    # Validate fields
    if not wo or not machine or not time or not status:
        rejected_records.append([line_number, line, "Missing field"])
        continue

    if not time[:2].isdigit() or not time[3:5].isdigit() or len(time) != 5 or time[2] != ":":
        rejected_records.append([line_number, line, "Invalid time format"])
        continue

    hour, minute = map(int, time.split(":"))

    if hour > 23 or minute > 59:
        rejected_records.append([line_number, line, "Invalid time value"])
        continue

    if status not in ["Completed", "Pending"]:
        rejected_records.append([line_number, line, "Invalid status"])
        continue

    valid_records.append([wo, machine, time, status])


# STEP 3: Aggregate records
total = len(valid_records)
completed = sum(1 for r in valid_records if r[3] == "Completed")
pending = sum(1 for r in valid_records if r[3] == "Pending")

# Count records machine-wise
machine_count = {}

for record in valid_records:
    machine = record[1]
    machine_count[machine] = machine_count.get(machine, 0) + 1


# STEP 4: Write human-readable report
with open(report_file, "w") as file:

    file.write("========== SHIFT LOG REPORT ==========\n\n")

    file.write(f"{'WO':<10}{'Machine':<15}{'Time':<12}{'Status':<12}\n")
    file.write("-" * 49 + "\n")

    for record in valid_records:
        file.write(
            f"{record[0]:<10}{record[1]:<15}"
            f"{record[2]:<12}{record[3]:<12}\n"
        )

    file.write("\n========== SUMMARY ==========\n")
    file.write(f"Total Valid Records : {total}\n")
    file.write(f"Completed Records   : {completed}\n")
    file.write(f"Pending Records     : {pending}\n")

    file.write("\nMachine-wise Records:\n")

    for machine, count in machine_count.items():
        file.write(f"{machine}: {count}\n")


# STEP 5: Write rejected records into CSV
with open(reject_file, "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow(["Line Number", "Rejected Line", "Reason"])

    writer.writerows(rejected_records)


# STEP 6: Display result
print("Pipeline completed successfully!")
print("Total input lines:", len(lines))
print("Valid records:", total)
print("Rejected records:", len(rejected_records))
print("Report saved to:", report_file)
print("Rejects saved to:", reject_file)


Pipeline completed successfully!
Total input lines: 15
Valid records: 12
Rejected records: 3
Report saved to: data/report.txt
Rejects saved to: data/rejects.csv


In [17]:

import re

# Read input file
with open("data/shift_log.txt", "r") as file:
    lines = file.readlines()

valid_records = []
rejected_records = []

# Regular expression pattern
pattern = r"^\s*(WO\d{2})\s*,\s*([^,]+)\s*,\s*(\d{2}:\d{2})\s*,\s*(Completed|Pending)\s*$"

regex_count = 0
split_count = 0

for line in lines:

    line = line.strip()

    if not line:
        continue

    # OLD METHOD: Using split()
    # parts = [p.strip() for p in line.split(",")]
    # if len(parts) == 4:
    #     split_count += 1

    # NEW METHOD: Using Regular Expression
    match = re.match(pattern, line)

    if match:
        wo, machine, time, status = match.groups()

        # Additional time validation
        hour, minute = map(int, time.split(":"))

        if hour <= 23 and minute <= 59:
            valid_records.append([wo, machine, time, status])
            regex_count += 1
        else:
            rejected_records.append(line)

    else:
        rejected_records.append(line)


# Display valid records
print("VALID RECORDS")
print("-" * 55)

for record in valid_records:
    print(f"{record[0]:<10} {record[1]:<15} {record[2]:<10} {record[3]}")

# Display rejected records
print("\nREJECTED RECORDS")
print("-" * 55)

for record in rejected_records:
    print(record)

# Comparison
print("\nPARSER COMPARISON")
print("Regex parsed records:", regex_count)
print("Old split method's basic field-count matches:", split_count)

print("\nRegex successfully validates the complete record format.")


VALID RECORDS
-------------------------------------------------------
WO01       Machine A       08:00      Completed
WO02       Machine B       08:15      Completed
WO03       Machine C       08:30      Pending
WO04       Machine A       08:45      Completed
WO05       Machine B       09:00      Completed
WO06       Machine C       09:15      Pending
WO07       Machine A       09:30      Completed
WO08       Machine B       09:45      Completed
WO09       Machine C       10:00      Completed
WO10       Machine A       10:15      Pending
WO11       Machine B       10:30      Completed
WO12       Machine C       10:45      Completed
WO15                       11:30      Completed

REJECTED RECORDS
-------------------------------------------------------
WO13, Machine A, INVALID_TIME, Completed
WO14, Machine B, 11:15,

PARSER COMPARISON
Regex parsed records: 13
Old split method's basic field-count matches: 0

Regex successfully validates the complete record format.


In [18]:

import os
import csv
import re
from datetime import datetime

# Folder and file paths
data_folder = "data"
history_file = os.path.join(data_folder, "run_history.csv")

os.makedirs(data_folder, exist_ok=True)

# Regex pattern for validating records
pattern = r"^\s*(WO\d{2})\s*,\s*([^,]+)\s*,\s*(\d{2}:\d{2})\s*,\s*(Completed|Pending)\s*$"

# Combined plant summary
plant_total = 0
plant_completed = 0
plant_pending = 0
plant_rejected = 0

# Store run history
history = []

# Process every TXT file in the folder
for filename in os.listdir(data_folder):

    if filename.endswith(".txt"):

        filepath = os.path.join(data_folder, filename)

        valid_records = []
        rejected_records = []

        # Read input file
        with open(filepath, "r") as file:
            lines = file.readlines()

        # Parse and validate records
        for line_number, line in enumerate(lines, start=1):

            match = re.match(pattern, line)

            if match:
                wo, machine, time, status = match.groups()

                hour, minute = map(int, time.split(":"))

                if hour <= 23 and minute <= 59:
                    valid_records.append([wo, machine, time, status])
                else:
                    rejected_records.append([line_number, line.strip(), "Invalid time"])
            else:
                rejected_records.append([line_number, line.strip(), "Invalid record format"])

        # Calculate individual statistics
        total = len(valid_records)
        completed = sum(1 for r in valid_records if r[3] == "Completed")
        pending = sum(1 for r in valid_records if r[3] == "Pending")
        rejects = len(rejected_records)

        # Generate individual report
        report_name = filename.replace(".txt", "_report.txt")
        report_path = os.path.join(data_folder, report_name)

        with open(report_path, "w") as report:

            report.write("========== FILE REPORT ==========\n")
            report.write("File Name: " + filename + "\n\n")

            report.write(f"{'WO':<10}{'Machine':<15}{'Time':<12}{'Status':<12}\n")
            report.write("-" * 49 + "\n")

            for r in valid_records:
                report.write(
                    f"{r[0]:<10}{r[1]:<15}{r[2]:<12}{r[3]:<12}\n"
                )

            report.write("\n---------- SUMMARY ----------\n")
            report.write(f"Valid Records: {total}\n")
            report.write(f"Completed: {completed}\n")
            report.write(f"Pending: {pending}\n")
            report.write(f"Rejected: {rejects}\n")

            report.write("\n---------- REJECTED RECORDS ----------\n")

            for r in rejected_records:
                report.write(f"{r[0]}: {r[1]} | Reason: {r[2]}\n")

        # Update combined plant totals
        plant_total += total
        plant_completed += completed
        plant_pending += pending
        plant_rejected += rejects

        # Add history entry
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        history.append([
            timestamp,
            filename,
            total,
            rejects
        ])

        print("Processed:", filename)


# Generate combined plant summary
summary_path = os.path.join(data_folder, "plant_summary.txt")

with open(summary_path, "w") as summary:

    summary.write("========== PLANT SUMMARY ==========\n\n")
    summary.write(f"Total Valid Records: {plant_total}\n")
    summary.write(f"Completed Records: {plant_completed}\n")
    summary.write(f"Pending Records: {plant_pending}\n")
    summary.write(f"Rejected Records: {plant_rejected}\n")


# Append entries to run history CSV
file_exists = os.path.exists(history_file)

with open(history_file, "a", newline="") as file:

    writer = csv.writer(file)

    # Write header only for a new file
    if not file_exists or os.path.getsize(history_file) == 0:
        writer.writerow([
            "Timestamp",
            "File Name",
            "Records Processed",
            "Rejects"
        ])

    writer.writerows(history)


# Final output
print("\n========== BATCH JOB COMPLETED ==========")
print("Total valid records:", plant_total)
print("Total rejected records:", plant_rejected)
print("Individual reports generated successfully.")
print("Plant summary:", summary_path)
print("Run history:", history_file)


Processed: plant_summary_report_report_report.txt
Processed: plant_summary_report_report.txt
Processed: plant_summary_report.txt
Processed: report_report_report_report_report.txt
Processed: report_report.txt
Processed: shift_log_report.txt
Processed: report_report_report_report.txt
Processed: shift_log_report_report_report_report.txt
Processed: plant_summary.txt
Processed: shift_log_report_report_report.txt
Processed: report_report_report.txt
Processed: shift_log.txt
Processed: shift_log_report_report.txt
Processed: report.txt

========== BATCH JOB COMPLETED ==========
Total valid records: 13
Total rejected records: 554
Individual reports generated successfully.
Plant summary: data/plant_summary.txt
Run history: data/run_history.csv
